# Alfido Tech — Superstore Sales Analysis

**Goal:** analyze sales performance across regions, categories and time; identify best/worst performing products and seasonality; recommend tactical improvements.

**Dataset:** `superstore_final_dataset__1_.csv` — 9,800 order line items, 2015–2018 (source: [Kaggle Superstore Sales](https://www.kaggle.com/datasets/bhanupratapbiswas/superstore-sales)).

**Note on scope:** this extract contains `Sales` only — no `Profit`, `Discount`, or `Quantity` fields, and no web/traffic data. Profit margin and true conversion-rate KPIs are therefore **not computable** from this file; every KPI below is revenue-based. Average order value and ship-time are used as the closest available efficiency proxies.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25
plt.rcParams['font.size'] = 10

PALETTE = ['#3fb9a8', '#e0a55c', '#d97463', '#8b8fd6', '#6fa8dc', '#c9b458']

df = pd.read_csv('superstore_final_dataset.csv', encoding='latin1')
print(df.shape)
df.head()

## 1. Clean & prepare

In [ ]:
# Duplicates & missing values
print("Duplicate rows:", df.duplicated().sum())
print(df.isna().sum())

In [ ]:
# Postal_Code has a few missing values (11) - not used in analysis, left as-is.
# Dates are day/month/year (verified: first component goes up to 31, second only to 12)
df['Order_Date'] = pd.to_datetime(df['Order_Date'], format='%d/%m/%Y')
df['Ship_Date']  = pd.to_datetime(df['Ship_Date'],  format='%d/%m/%Y')

df['Ship_Days']  = (df['Ship_Date'] - df['Order_Date']).dt.days
df['Order_Year'] = df['Order_Date'].dt.year
df['Order_Month'] = df['Order_Date'].dt.month
df['Order_YM']   = df['Order_Date'].dt.to_period('M').astype(str)

assert (df['Ship_Days'] >= 0).all(), "found negative ship times"
print("Date range:", df['Order_Date'].min().date(), "to", df['Order_Date'].max().date())
print("Unique orders:", df['Order_ID'].nunique(), "| customers:", df['Customer_ID'].nunique(), "| products:", df['Product_ID'].nunique())

## 2. KPIs

In [ ]:
total_revenue = df['Sales'].sum()
n_orders      = df['Order_ID'].nunique()
n_customers   = df['Customer_ID'].nunique()
n_products    = df['Product_ID'].nunique()
aov           = total_revenue / n_orders
avg_ship_days = df['Ship_Days'].mean()
rev_per_customer = total_revenue / n_customers

kpis = pd.Series({
    'Total revenue ($)': round(total_revenue, 2),
    'Orders': n_orders,
    'Customers': n_customers,
    'Distinct products sold': n_products,
    'Average order value ($)': round(aov, 2),
    'Revenue per customer ($)': round(rev_per_customer, 2),
    'Average ship time (days)': round(avg_ship_days, 2),
})
kpis

**Conversion metrics:** not available — the dataset has no site-visit, cart, or lead data, so true conversion rate can't be derived. Orders-per-customer (below) is the closest repeat-purchase proxy.

In [ ]:
orders_per_customer = df.groupby('Customer_ID')['Order_ID'].nunique()
print("Median orders per customer:", orders_per_customer.median())
print("Customers with 2+ orders:", (orders_per_customer >= 2).sum(), f"({(orders_per_customer>=2).mean():.1%})")

## 3. Revenue trend over time

In [ ]:
by_year = df.groupby('Order_Year')['Sales'].sum()
yoy = by_year.pct_change() * 100
print(by_year.round(0))
print("\nYoY growth %:")
print(yoy.round(1))

In [ ]:
ts = df.groupby('Order_YM')['Sales'].sum()

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(ts.index, ts.values, color=PALETTE[0], linewidth=2)
ax.fill_between(range(len(ts)), ts.values, color=PALETTE[0], alpha=0.12)
ax.set_title('Monthly revenue, 2015-2018')
ax.set_ylabel('Revenue ($)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, p: f'${v/1000:.0f}K'))
ax.set_xticks(range(0, len(ts), 3))
ax.set_xticklabels(ts.index[::3], rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 4. Seasonality

In [ ]:
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
monthly = df.groupby('Order_Month')['Sales'].sum().reindex(range(1,13))

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(month_names, monthly.values, color=PALETTE[2])
ax.set_title('Total revenue by calendar month (all years combined)')
ax.set_ylabel('Revenue ($)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, p: f'${v/1000:.0f}K'))
plt.tight_layout()
plt.show()

print("Peak months:", monthly.sort_values(ascending=False).head(3).index.map(lambda m: month_names[m-1]).tolist())
print("Weakest months:", monthly.sort_values().head(3).index.map(lambda m: month_names[m-1]).tolist())

Clear Q4 (and September) seasonality: November, December and September are the three strongest months, while January and February are consistently the weakest — roughly a 5-6x swing between trough and peak.

## 5. Category & sub-category performance

In [ ]:
cat = df.groupby('Category')['Sales'].agg(revenue='sum', orders='count').sort_values('revenue', ascending=False)
cat['avg_line_value'] = (cat['revenue'] / cat['orders']).round(2)
cat

In [ ]:
subcat = df.groupby('Sub_Category')['Sales'].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
colors = [PALETTE[0] if v >= subcat.median() else PALETTE[2] for v in subcat.values]
ax.barh(subcat.index[::-1], subcat.values[::-1], color=colors[::-1])
ax.set_title('Revenue by sub-category')
ax.set_xlabel('Revenue ($)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, p: f'${v/1000:.0f}K'))
plt.tight_layout()
plt.show()

## 6. Best & worst performing products

In [ ]:
product_rev = df.groupby('Product_Name')['Sales'].sum().sort_values(ascending=False)

print("Top 10 products by revenue:")
display(product_rev.head(10))
print("\nBottom 10 products by revenue:")
display(product_rev.tail(10))

In [ ]:
top10 = product_rev.head(10)
labels = [n[:38] + ('...' if len(n) > 38 else '') for n in top10.index]

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(labels[::-1], top10.values[::-1], color=PALETTE[1])
ax.set_title('Top 10 products by revenue')
ax.set_xlabel('Revenue ($)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, p: f'${v/1000:.0f}K'))
plt.tight_layout()
plt.show()

## 7. Regional performance

In [ ]:
region = df.groupby('Region')['Sales'].agg(revenue='sum', orders='count').sort_values('revenue', ascending=False)
region['avg_line_value'] = (region['revenue'] / region['orders']).round(2)
region

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].bar(region.index, region['revenue'], color=PALETTE[3])
axes[0].set_title('Revenue by region')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, p: f'${v/1000:.0f}K'))

state = df.groupby('State')['Sales'].sum().sort_values(ascending=False).head(10)
axes[1].barh(state.index[::-1], state.values[::-1], color=PALETTE[4])
axes[1].set_title('Top 10 states by revenue')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, p: f'${v/1000:.0f}K'))

plt.tight_layout()
plt.show()

In [ ]:
pivot = df.pivot_table(index='Region', columns='Category', values='Sales', aggfunc='sum')

fig, ax = plt.subplots(figsize=(7, 4))
im = ax.imshow(pivot.values, cmap='YlGnBu', aspect='auto')
ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(len(pivot.index))); ax.set_yticklabels(pivot.index)
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        ax.text(j, i, f"${pivot.values[i,j]/1000:.0f}K", ha='center', va='center', fontsize=9)
ax.set_title('Revenue: Region x Category')
plt.colorbar(im, ax=ax, label='Revenue ($)')
plt.tight_layout()
plt.show()

## 8. Segment & fulfillment

In [ ]:
seg = df.groupby('Segment')['Sales'].sum().sort_values(ascending=False)
shipmode = df.groupby('Ship_Mode').agg(revenue=('Sales','sum'), orders=('Sales','count'), avg_ship_days=('Ship_Days','mean')).sort_values('revenue', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].pie(seg.values, labels=seg.index, autopct='%1.0f%%', colors=PALETTE[:3])
axes[0].set_title('Revenue share by segment')

axes[1].bar(shipmode.index, shipmode['revenue'], color=PALETTE[5])
axes[1].set_title('Revenue by ship mode')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, p: f'${v/1000:.0f}K'))
axes[1].tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

shipmode.round(2)

## 9. Key findings

- **Growth:** revenue dipped -4.3% in 2016, then grew +30.6% in 2017 and +20.3% in 2018 — momentum is real and recent.
- **Seasonality:** Nov/Dec/Sep are consistently the strongest months; Jan/Feb the weakest, by a 5-6x margin.
- **Category:** Technology ($827K) and Furniture ($729K) lead; Office Supplies has the most orders (5,909) but the lowest average line value.
- **Sub-category concentration:** Phones and Chairs alone generate over $650K — more than the bottom nine sub-categories combined. Fasteners, Labels and Envelopes each bring in under $17K.
- **Regional gap:** West ($710K) and East ($670K) outsell South ($389K) by roughly 1.8x, a gap consistent across the whole period.
- **Segment:** Consumer ($1.15M) outsells Corporate and Home Office combined.
- **Data gap:** no Profit/Discount/Quantity fields — profit margin and unit economics can't be computed from this extract.

## 10. Five tactical recommendations for Alfido Tech

1. **Protect the Nov-Dec peak and smooth the Jan-Feb trough** — pre-position inventory and staffing for Q4, and run post-peak retention offers to pull some of that demand into the new year.
2. **Double down on Phones and Chairs** — feature them in bundles and promotions; they already outperform everything else per line.
3. **Prune or bundle the long-tail sub-categories** (Fasteners, Labels, Envelopes) — treat them as checkout add-ons rather than standalone marketing lines.
4. **Run a targeted South-region growth push** — the region has trailed West and East across all four years; a local promotion or partner push could close a persistent, structural gap.
5. **Extend what works in Consumer to Corporate and Home Office** — those two segments show the most unclaimed headroom; adapt Consumer-proven bundles/financing for procurement and home-office buyers.

*Recommend collecting Profit/Discount/Quantity data going forward — margin-aware versions of these same cuts would sharpen every one of these calls.*